In [1]:
import pandas as pd
import numpy as np
import umap
import glob
import os
import joblib
from pathlib import Path
import pickle
from scipy.io import readsav

In [2]:
SET_JPARAM = True
SAVE_DIR = Path(Path.home(),'ECM_mod_output/UMAP/ODE_embeddings')
RED_DIR = Path(Path.home(),'ECM_mod_output/UMAP')
if not os.path.exists(SAVE_DIR):
    os.mkdir(SAVE_DIR)

In [3]:
FILE_LOC = Path(Path.home(),'ECM_mod_output/ODE_output')
if SET_JPARAM:
    filenames = [Path(FILE_LOC,f) for f in os.listdir(FILE_LOC) if  'psoparam' in f]
else:
    filenames = [Path(FILE_LOC,f) for f in os.listdir(FILE_LOC) if  'range' in f]    
    
all_odes = sorted(filenames)

In [4]:
all_odes

[PosixPath('/Users/darren/ECM_mod_output/ODE_output/ERL_psoparam_0_000.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_output/ERL_psoparam_0_001.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_output/ERL_psoparam_0_002.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_output/ERL_psoparam_0_003.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_output/ERL_psoparam_0_004.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_output/ERL_psoparam_0_005.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_output/ERL_psoparam_0_006.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_output/ERL_psoparam_0_007.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_output/ERL_psoparam_0_008.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_output/ERL_psoparam_0_009.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_output/ERL_psoparam_0_010.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_output/ERL_psoparam_0_011.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_output/ERL_psoparam

In [5]:
reducer_fn = [f for f in os.listdir(RED_DIR) if 'reducer' in f]
reducer_fn

['reducer_03_20_0.1_0.1_4_0.5.sav',
 'reducer_psoparams_20240601_0.1_0.1_4_0.5.sav']

In [6]:
# Important params to set the window size and step time distance
init_dt = 0.1                # initial dt from the raw model output
dt = 0.1                      # dt for umap window
window = 4                    # window size in hours
time_step = 0.5               # time step between windows

#umap params
neighbs = 50
min_d = 0.05

#Load umap reducer object
# reducer = joblib.load('C:\\Users\\olesk\\data\\reducer_07_25_0.1_0.1_4_0.5.sav')
# reducer = joblib.load('C:\\Users\\olesk\\ECM_mod_output\\reducer_03_20_0.1_0.1_4_0.5.sav')

if SET_JPARAM:
    rname = [Path(RED_DIR, f) for f in reducer_fn if 'psoparams' in f][0]
else:
    rname = [Path(RED_DIR, f) for f in reducer_fn if 'psoparams' not in f][0]
    # rname = Path(RED_DIR, 'reducer_03_20_0.1_0.1_4_0.5.sav')

reducer = joblib.load(rname)

In [7]:
STOCH_LOC = Path(Path.home(),'ECM_mod_output/')
if SET_JPARAM:
    filenames = [Path(STOCH_LOC,f) for f in os.listdir(STOCH_LOC) if 'stoch_sim' in f and 'psoparam' in f]
else:
    filenames = [Path(STOCH_LOC,f) for f in os.listdir(STOCH_LOC) if 'stoch_sim' in f and 'cycdi_0_0' in f]
filenames = sorted(filenames)

filenames_conds = [filenames[2], filenames[3], filenames[0],
                filenames[1], filenames[4], filenames[5]]

filenames_conds

[PosixPath('/Users/darren/ECM_mod_output/stoch_sim_soft_DMSO_psoparam_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/stoch_sim_soft_erl_psoparam_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/stoch_sim_int_DMSO_psoparam_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/stoch_sim_int_erl_psoparam_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/stoch_sim_stiff_DMSO_psoparam_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/stoch_sim_stiff_erl_psoparam_cycdi_0_0.csv')]

In [8]:
if SET_JPARAM:
    filenames_ode = [Path(STOCH_LOC,f) for f in os.listdir(STOCH_LOC) if 'ODE_sim' in f and 'psoparam' in f]
else:
    filenames_ode = [Path(STOCH_LOC,f) for f in os.listdir(STOCH_LOC) if 'ODE_sim' in f and 'cycdi_0_0' in f]
    
filenames_ode = sorted(filenames_ode)

filenames_conds_ode = [filenames_ode[2], filenames_ode[3], filenames_ode[0],
                filenames_ode[1], filenames_ode[4], filenames_ode[5]]

filenames_conds_ode


[PosixPath('/Users/darren/ECM_mod_output/ODE_sim_soft_DMSO_psoparam_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_sim_soft_erl_psoparam_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_sim_int_DMSO_psoparam_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_sim_int_erl_psoparam_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_sim_stiff_DMSO_psoparam_cycdi_0_0.csv'),
 PosixPath('/Users/darren/ECM_mod_output/ODE_sim_stiff_erl_psoparam_cycdi_0_0.csv')]

In [9]:
def calc_minmax(filenames, filenames_ode):
    max_prots=[0]*18
    min_prots=[1000]*18
    
    for f in filenames:
        tmp_max=[]
        tmp_min=[]
        prot_tmp = pd.read_csv(f, sep=",", header=None, usecols = range(0,19))
        prot_tmp.columns = ["skp2", "p21", "pp21", "cdh1_act", "cdh1_inact", "myc", "cycD", "cycD-p21", 
        "Rb", "pRb", "E2F", "E2F-Rb", "cycE", "cycE-p21", "cycA", "cycA-p21",
        "cycB", "cycB-p21", "cdc20"]
        prot_tmp = prot_tmp.drop(columns=['cdh1_inact'])
        #prot_tmp = prot_tmp/200
        prot_tmp[prot_tmp<0] = 0 #from old extrap. method, don't think I need this
        
        tmp_max = [prot_tmp[col].max() for col in prot_tmp]
        tmp_min = [prot_tmp[col].min() for col in prot_tmp]
        
        max_prots = [max(max_prots[n], tmp_max[n]) for n in range(18)]
        min_prots = [min(min_prots[n], tmp_min[n]) for n in range(18)]
        
    for f in filenames_ode:
        tmp_max=[]
        tmp_min=[]
        prot_tmp_ode = pd.read_csv(f, sep=",", header=None, usecols = range(0,19))
        prot_tmp_ode.columns = ["skp2", "p21", "pp21", "cdh1_act", "cdh1_inact", "myc", "cycD", "cycD-p21", 
        "Rb", "pRb", "E2F", "E2F-Rb", "cycE", "cycE-p21", "cycA", "cycA-p21",
        "cycB", "cycB-p21", "cdc20"]
        prot_tmp_ode = prot_tmp_ode.drop(columns=['cdh1_inact'])
        prot_tmp[prot_tmp<0] = 0 #from old extrap. method, don't think I need this
        
        tmp_max = [prot_tmp_ode[col].max() for col in prot_tmp_ode]
        tmp_min = [prot_tmp_ode[col].min() for col in prot_tmp_ode]
        
        max_prots = [max(max_prots[n], tmp_max[n]) for n in range(18)]
        min_prots = [min(min_prots[n], tmp_min[n]) for n in range(18)]  
        
    return max_prots, min_prots    

In [10]:
max_prots, min_prots = calc_minmax(filenames, filenames_ode)

In [11]:

win_len = int(window/dt)
all_prot_tmp = np.empty((0,win_len*18))     #careful, this could end up being massive 
all_prot_tmp_ode = np.empty((0,win_len*18))
all_time_tmp = np.empty((0,win_len))
all_time_at_tmp = np.empty((0,1))
#range_ind = np.arange(int(win_len/2),int(len(time_tmp2)-(int(win_len/2))),int(time_step/dt))

for f in all_odes:
    prot_tmp = pd.read_csv(f, sep=",", header=None, usecols = range(0,19))
    prot_tmp.columns = ["skp2", "p21", "pp21", "cdh1_act", "cdh1_inact", "myc", "cycD", "cycD-p21", 
    "Rb", "pRb", "E2F", "E2F-Rb", "cycE", "cycE-p21", "cycA", "cycA-p21",
    "cycB", "cycB-p21", "cdc20"]
    prot_tmp = prot_tmp.drop(columns=['cdh1_inact'])
    
    range_ind = np.arange(int(win_len/2),int(len(prot_tmp)-(int(win_len/2))),int(time_step/dt))
    #prot_tmp = prot_tmp/200
    for ii in range(18):
        prot_tmp.iloc[:,ii] = (prot_tmp.iloc[:,ii] - min_prots[ii])/(max_prots[ii] - min_prots[ii])

    prot_re_tmp = np.empty((len(range_ind),win_len*18))
    time_re_tmp = np.empty((len(range_ind),win_len))
    time_at_tmp = np.empty((len(range_ind),1))

    ii=0
    for n in range(int(win_len/2),int(len(prot_tmp)-(int(win_len/2))),int(time_step/dt)):
        tmp = prot_tmp.iloc[n-int(win_len/2):n+int(win_len/2)]
        #tmp_t = time_tmp2['time'].iloc[n-int(win_len/2):n+int(win_len/2)]
        tmp=tmp.to_numpy()
        tmp=np.reshape(np.transpose(tmp),win_len*18)
        #tmp_at = time_tmp2['time'].iloc[n]

        prot_re_tmp[ii,:] = np.array([tmp])
        #time_re_tmp[ii,:] = np.array([tmp_t])
        #time_at_tmp[ii,:] = np.array([[tmp_at]])
        ii+=1

    embedding_tmp = reducer.transform(prot_re_tmp)
    absolute_path = f.absolute().as_posix()    
    split_s = absolute_path.split('.')
    split_ss = split_s[0].split('/')
    name = 'embedding_'+split_ss[-1]+'.csv'
    
    np.savetxt(Path(SAVE_DIR, name), embedding_tmp, delimiter=',', fmt='%f')



